# QK-Norm 与 Logit Softcapping：看见注意力饱和与 Mask 顺序

**面试问题：Q/K 归一化、温度和 softcap 分别解决什么，怎样保证不破坏 causal mask？**

## 回答主线

用一个可读任务先建立朴素 baseline，再从基础算子实现核心机制，输出中间状态、指标对照和失败修正。断言只在最后保护最关键的不变量；受控小数据用于解释机制，不冒充真实基础模型质量。

## 真实案例

客服模型正在阅读“退款政策—订单状态—用户问题”六个 token。某个 query/key 通道出现幅度异常，原始点积把 softmax 压成近乎 one-hot，梯度和混合精度都不稳定。案例展示原始 logits、QK-Norm、可学习温度和 tanh softcap 的逐步变化，并故意把 softcap 放在 mask 之后，观察未来 token 如何重新获得非零概率。

### 输入预览：六个政策与订单 token

In [1]:
import numpy as np  # 导入矩阵运算以从零实现注意力稳定化。

tokens = ["<系统>", "退款政策", "订单A7842", "已签收", "用户询问", "多久到账"]  # 构造具有真实客服语义的六个 token。
queries = np.array([[0.3, 0.2, -0.1, 0.4], [0.4, -0.2, 0.5, 0.1], [0.2, 0.1, 0.3, -0.2], [0.5, 0.4, -0.3, 0.2], [0.4, 0.2, 0.1, 0.3], [14.0, 0.3, -0.2, 0.1]], dtype=float)  # 给最后 query 注入一个异常大通道。
keys = np.array([[0.7, 0.1, -0.2, 0.3], [1.1, 0.2, 0.1, -0.1], [0.4, -0.3, 0.8, 0.2], [0.2, 0.5, -0.4, 0.6], [0.1, 0.2, 0.3, 0.4], [0.3, -0.1, 0.2, 0.5]], dtype=float)  # 构造六个 key 表示。
print("token 与向量范数：")  # 输出幅度异常的输入预览。
for token, query, key in zip(tokens, queries, keys):  # 逐 token 展示 Q/K 范数。
    print(f"{token:<10} ||Q||={np.linalg.norm(query):6.3f} ||K||={np.linalg.norm(key):6.3f}")  # 显示最后 query 范数远大于其余位置。

token 与向量范数：
<系统>       ||Q||= 0.548 ||K||= 0.794
退款政策       ||Q||= 0.678 ||K||= 1.127
订单A7842    ||Q||= 0.424 ||K||= 0.964
已签收        ||Q||= 0.735 ||K||= 0.900
用户询问       ||Q||= 0.548 ||K||= 0.548
多久到账       ||Q||=14.005 ||K||= 0.624


## Baseline 基线：原始缩放点积注意力

In [2]:
def softmax(vector):  # 实现数值稳定的一维 softmax。
    shifted = vector - np.max(vector)  # 减去最大值避免指数溢出。
    exponentials = np.exp(shifted)  # 计算平移后的指数权重。
    return exponentials / exponentials.sum()  # 返回和为一的注意力概率。

raw_logits = queries[-1] @ keys.T / np.sqrt(queries.shape[-1])  # 计算最后 token 对全部 key 的原始缩放点积。
raw_probabilities = softmax(raw_logits)  # 将原始 logits 转成注意力概率。
raw_entropy = -float(np.sum(raw_probabilities * np.log(raw_probabilities + 1e-12)))  # 计算注意力熵衡量饱和程度。
print("原始 logits：", np.round(raw_logits, 3))  # 展示异常 Q 通道如何放大部分 key。
print("原始概率：", np.round(raw_probabilities, 4))  # 展示 softmax 是否接近 one-hot。
print(f"原始注意力熵={raw_entropy:.4f}，最大概率={raw_probabilities.max():.2%}")  # 用两个可解释指标量化饱和。

原始 logits： [4.95  7.715 2.685 1.545 0.72  2.09 ]
原始概率： [5.850e-02 9.293e-01 6.100e-03 1.900e-03 9.000e-04 3.400e-03]
原始注意力熵=0.3025，最大概率=92.93%


### 核心实现：QK-Norm、温度与 Softcap

In [3]:
def l2_normalize(matrix, epsilon=1e-8):  # 实现逐 token 的 L2 Q/K 归一化。
    norms = np.sqrt(np.sum(matrix ** 2, axis=-1, keepdims=True) + epsilon)  # 计算稳定向量范数。
    return matrix / norms  # 把每个 token 表示缩放到单位球面。

def softcap(logits, cap):  # 实现保持单调且有界的 tanh logit softcap。
    return cap * np.tanh(logits / cap)  # 把任意大 logits 压到正负 cap 范围。

normalized_queries = l2_normalize(queries)  # 对全部 query 执行逐 token 归一化。
normalized_keys = l2_normalize(keys)  # 对全部 key 执行逐 token 归一化。
temperature = 3.0  # 设置可学习温度的教学取值恢复适当区分度。
normalized_logits = temperature * (normalized_queries[-1] @ normalized_keys.T)  # 计算归一化余弦相似度 logits。
capped_logits = softcap(normalized_logits, cap=2.0)  # 对归一化 logits 再应用有界 softcap。
capped_probabilities = softmax(capped_logits)  # 计算稳定化后的注意力概率。
capped_entropy = -float(np.sum(capped_probabilities * np.log(capped_probabilities + 1e-12)))  # 计算稳定化后的注意力熵。
print("QK-Norm logits：", np.round(normalized_logits, 3))  # 展示归一化后幅度不再由异常范数主导。
print("Softcap logits：", np.round(capped_logits, 3))  # 展示 logits 被限制在正负二附近。
print("稳定化概率：", np.round(capped_probabilities, 4))  # 展示概率仍有区分但不过度饱和。
print(f"稳定化熵={capped_entropy:.4f}，最大概率={capped_probabilities.max():.2%}")  # 对比机制对概率分布的影响。

QK-Norm logits： [2.672 2.933 1.193 0.735 0.563 1.434]
Softcap logits： [1.741 1.798 1.069 0.704 0.549 1.23 ]
稳定化概率： [0.2614 0.2765 0.1334 0.0926 0.0793 0.1567]
稳定化熵=1.6868，最大概率=27.65%


## 结果解读：三个组件解决不同问题

In [4]:
comparison = [  # 构造原始、仅归一化和归一化加 softcap 的对照记录。
    ("原始点积", raw_logits, raw_probabilities),  # 原始方案受向量范数影响。
    ("QK-Norm", normalized_logits, softmax(normalized_logits)),  # 归一化方案移除范数自由度。
    ("QK-Norm+Softcap", capped_logits, capped_probabilities),  # softcap 进一步限制极端温度输出。
]  # 完成对照表数据。
print("方案               logit范围        max_prob  entropy")  # 输出对照表表头。
for name, logits, probabilities in comparison:  # 逐方案计算可解释统计量。
    entropy = -float(np.sum(probabilities * np.log(probabilities + 1e-12)))  # 计算当前方案注意力熵。
    print(f"{name:<18} [{logits.min():6.2f},{logits.max():6.2f}] {probabilities.max():8.2%} {entropy:8.4f}")  # 展示幅度、集中度和熵。
print("解读：QK-Norm 控制向量范数；temperature 控制可学习区分度；softcap 只限制极端 logits，三者不能替代 mask。")  # 明确三个机制的职责边界。

方案               logit范围        max_prob  entropy
原始点积               [  0.72,  7.72]   92.93%   0.3025
QK-Norm            [  0.56,  2.93]   42.13%   1.4150
QK-Norm+Softcap    [  0.55,  1.80]   27.65%   1.6868
解读：QK-Norm 控制向量范数；temperature 控制可学习区分度；softcap 只限制极端 logits，三者不能替代 mask。


## 失败案例：先加 Mask 再 Softcap 会让未来位置泄漏

In [5]:
causal_allowed = np.array([True, True, True, True, False, False])  # 假设当前位置只能观察前四个 token。
correct_masked_logits = np.where(causal_allowed, softcap(normalized_logits, 2.0), -np.inf)  # 正确顺序是先稳定合法 logits 再写入负无穷 mask。
correct_masked_probabilities = softmax(correct_masked_logits)  # 计算未来位置严格为零的正确概率。
wrong_pre_masked = np.where(causal_allowed, normalized_logits, -1e9)  # 错误实现先用巨大负数写入 mask。
wrong_capped_logits = softcap(wrong_pre_masked, 2.0)  # softcap 把负十亿恢复成有限的负二。
wrong_probabilities = softmax(wrong_capped_logits)  # 有限负二使未来位置重新得到非零概率。
print("正确 mask 概率：", np.round(correct_masked_probabilities, 6))  # 展示未来两个位置严格为零。
print("错误顺序概率：", np.round(wrong_probabilities, 6))  # 展示被 softcap 恢复的未来泄漏。
print(f"错误未来概率总和={wrong_probabilities[~causal_allowed].sum():.4%}")  # 量化顺序错误造成的因果泄漏。

正确 mask 概率： [0.342122 0.361986 0.174648 0.121244 0.       0.      ]
错误顺序概率： [0.336658 0.356204 0.171859 0.119307 0.007986 0.007986]
错误未来概率总和=1.5972%


### 生产边界

In [6]:
attention_contract = {"qk_norm": "l2", "temperature": temperature, "softcap": 2.0, "operation_order": ["qk_norm", "dot", "temperature", "softcap", "mask", "softmax"], "dtype": "bf16"}  # 构造 checkpoint 与 kernel 必须绑定的稳定性合同。
print("Attention 合同：", attention_contract)  # 展示服务端不能猜测训练时操作顺序。
print("生产替换点：真实模型需按 head/layer 校准温度、验证混合精度 kernel、训练稳定性、长上下文质量和 fused mask 语义。")  # 明确 NumPy 单头实验与真实 Transformer 的差距。

Attention 合同： {'qk_norm': 'l2', 'temperature': 3.0, 'softcap': 2.0, 'operation_order': ['qk_norm', 'dot', 'temperature', 'softcap', 'mask', 'softmax'], 'dtype': 'bf16'}
生产替换点：真实模型需按 head/layer 校准温度、验证混合精度 kernel、训练稳定性、长上下文质量和 fused mask 语义。


## 回归测试：只保护范围、因果与职责

In [7]:
assert np.max(np.abs(capped_logits)) <= 2.0 + 1e-12  # 验证 softcap 后所有合法 logits 均处于配置范围。
assert capped_entropy > raw_entropy  # 验证当前异常输入下稳定化方案降低了过度饱和。
assert np.all(correct_masked_probabilities[~causal_allowed] == 0.0)  # 验证正确顺序严格阻断未来位置。
assert wrong_probabilities[~causal_allowed].sum() > 0.0  # 验证失败案例确实暴露 softcap 后 mask 泄漏。
assert attention_contract["operation_order"][-2:] == ["mask", "softmax"]  # 验证发布合同固定正确的最终操作顺序。
print("回归测试通过：softcap 范围、熵、causal mask、失败探针和操作合同均成立。")  # 用少量断言总结核心机制。

回归测试通过：softcap 范围、熵、causal mask、失败探针和操作合同均成立。
